# Model Training on Google Colab (GPU)

Trains both MAS models with LoRA fine-tuning on a cloud GPU:

- **Model 1 (Mistral-7B)**: RCA classification — structured signals → root cause label
- **Model 2 (Qwen-7B)**: RCA generation — evidence text → diagnosis + fix plan

**Requirements**: Colab with GPU runtime (T4 free tier works, A100 is faster)

### Setup Instructions
1. Upload this notebook to [Google Colab](https://colab.research.google.com/)
2. Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU
3. Upload your data files when prompted
4. Run all cells

## 0. Install Dependencies & Check GPU

In [ ]:
!pip install -q torch transformers peft trl accelerate bitsandbytes datasets sentencepiece protobuf pandas pyarrow

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 1. Upload Data

Upload the two parquet files from your local `data/processed/` directory.

In [ ]:
import os
from pathlib import Path

# Create directory structure
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("agents/models/trained/model1_mistral").mkdir(parents=True, exist_ok=True)
Path("agents/models/trained/model2_qwen").mkdir(parents=True, exist_ok=True)

# Upload files
from google.colab import files
print("Upload agent1_structured.parquet and agent2_evidence.parquet")
uploaded = files.upload()

for filename in uploaded:
    dest = f"data/processed/{filename}"
    os.rename(filename, dest)
    print(f"  → {dest}")

In [ ]:
import pandas as pd

agent1_df = pd.read_parquet("data/processed/agent1_structured.parquet")
agent2_df = pd.read_parquet("data/processed/agent2_evidence.parquet")
print(f"Agent 1: {len(agent1_df):,} rows, {len(agent1_df.columns)} cols")
print(f"Agent 2: {len(agent2_df):,} rows, {len(agent2_df.columns)} cols")
print(f"\nRoot cause distribution:")
print(agent1_df["root_cause_family"].value_counts())

---
## 2. Model 1: Mistral-7B LoRA — RCA Classification

| Parameter | Value |
|-----------|-------|
| Base model | `mistralai/Mistral-7B-Instruct-v0.3` |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Quantization | 4-bit NF4 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Max sequence length | 512 |

In [ ]:
import json
import warnings
import numpy as np
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore", category=UserWarning)

# ---------- Constants ----------
MODEL1_ID = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL1_OUTDIR = Path("agents/models/trained/model1_mistral")
MAX_SEQ_LEN_1 = 512

SYSTEM_PROMPT_1 = (
    "You are a Kubernetes Root Cause Analysis agent. Given structured "
    "observability signals from a Kubernetes incident, classify the root "
    "cause family. Respond with ONLY the root cause label."
)

In [ ]:
# ---------- Format data ----------
def format_incident_prompt(row):
    fields = [
        f"Namespace: {row.get('namespace', 'unknown')}",
        f"Workload Kind: {row.get('workload_kind', 'unknown')}",
        f"Pod Status: {row.get('pod_status', 'unknown')}",
        f"Waiting Reason: {row.get('waiting_reason', 'unknown')}",
        f"Error Message: {row.get('error_message', 'unknown')}",
        f"Event Type: {row.get('event_type', 'unknown')}",
        f"Event Reason: {row.get('event_reason', 'unknown')}",
        f"Event Message: {row.get('event_message', 'unknown')}",
        f"Restart Count: {row.get('restart_count', 0)}",
        f"OOM Killed: {row.get('oom_killed', False)}",
        f"Symptom Family: {row.get('symptom_family', 'unknown')}",
        f"Difficulty: {row.get('difficulty', 'unknown')}",
        f"Noise Level: {row.get('noise_level', 0)}",
    ]
    return "\n".join(fields)

def build_model1_chat(row):
    incident = format_incident_prompt(row)
    target = row.get("root_cause_family", "unknown")
    return (
        f"<s>[INST] {SYSTEM_PROMPT_1}\n\n"
        f"Classify the root cause of this Kubernetes incident:\n\n"
        f"{incident} [/INST] {target}</s>"
    )

# Prepare dataset
df1 = agent1_df.copy()
str_cols = df1.select_dtypes(include=["object"]).columns
df1[str_cols] = df1[str_cols].fillna("unknown")
df1 = df1.fillna(0)

texts1 = [build_model1_chat(r) for r in df1.to_dict("records")]
ds1 = Dataset.from_pandas(pd.DataFrame({"text": texts1}))
split1 = ds1.train_test_split(test_size=0.2, seed=42)
print(f"Model 1 — train: {len(split1['train']):,}, test: {len(split1['test']):,}")
print(f"Sample:\n{texts1[0][:300]}...")

In [ ]:
# ---------- Load Mistral + LoRA (tuned for higher accuracy) ----------
bnb_config_1 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer1 = AutoTokenizer.from_pretrained(MODEL1_ID, trust_remote_code=True)
if tokenizer1.pad_token is None:
    tokenizer1.pad_token = tokenizer1.eos_token
    tokenizer1.pad_token_id = tokenizer1.eos_token_id

model1 = AutoModelForCausalLM.from_pretrained(
    MODEL1_ID,
    quantization_config=bnb_config_1,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model1.config.use_cache = False

# Higher rank + more target modules = more capacity to learn
lora_config_1 = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model1 = get_peft_model(model1, lora_config_1)
trainable, total = model1.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ---------- Train Model 1 (tuned for 70%+ accuracy) ----------
training_args_1 = SFTConfig(
    output_dir=str(MODEL1_OUTDIR),
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    fp16=False,
    report_to="none",
    max_length=MAX_SEQ_LEN_1,
    dataset_text_field="text",
)

trainer1 = SFTTrainer(
    model=model1,
    processing_class=tokenizer1,
    train_dataset=split1["train"],
    eval_dataset=split1["test"],
    args=training_args_1,
)

print("Training Model 1 (Mistral-7B) — 8 epochs, LoRA r=64...")
print("This will take ~20-30 min on A100")
trainer1.train()

In [ ]:
# ---------- Save Model 1 ----------
adapter_path_1 = MODEL1_OUTDIR / "lora_adapter"
model1.save_pretrained(str(adapter_path_1))
tokenizer1.save_pretrained(str(adapter_path_1))

metrics1 = trainer1.evaluate()
with open(MODEL1_OUTDIR / "eval_metrics.json", "w") as f:
    json.dump(metrics1, f, indent=2)

metadata1 = {
    "model_id": MODEL1_ID, "task": "rca_classification",
    "target": "root_cause_family",
    "labels": list(agent1_df["root_cause_family"].dropna().unique()),
    "lora_r": 16, "lora_alpha": 32, "epochs": 3,
    "dataset_rows": len(ds1),
}
with open(MODEL1_OUTDIR / "model1_metadata.json", "w") as f:
    json.dump(metadata1, f, indent=2)

print(f"\nModel 1 saved to {adapter_path_1}")
print(f"Eval loss: {metrics1.get('eval_loss', 'N/A'):.4f}")

In [ ]:
# ---------- Model 1 Accuracy Evaluation ----------
from collections import defaultdict

correct = 0
total_samples = 0
ground_truths = []
predictions_list = []

# Evaluate on 300 samples
eval_samples = agent1_df.sample(n=300, random_state=42).to_dict("records")

print("Running inference on 300 test samples...")
for i, row in enumerate(eval_samples):
    incident_text = format_incident_prompt(row)
    prompt = (
        f"<s>[INST] {SYSTEM_PROMPT_1}\n\n"
        f"Classify the root cause of this Kubernetes incident:\n\n"
        f"{incident_text} [/INST]"
    )
    inputs = tokenizer1(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN_1)
    inputs = {k: v.to(model1.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model1.generate(**inputs, max_new_tokens=20, temperature=0.1, do_sample=False)
    pred = tokenizer1.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()

    gt = str(row.get("root_cause_family", "unknown")).lower()
    ground_truths.append(gt)
    predictions_list.append(pred)

    if gt in pred or pred in gt:
        correct += 1

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/300 — running accuracy: {correct/(i+1)*100:.1f}%")

accuracy = correct / len(eval_samples) * 100
print(f"\n{'='*50}")
print(f"ACCURACY: {correct}/{len(eval_samples)} = {accuracy:.1f}%")
print(f"{'='*50}")

# Per-class breakdown
class_correct = defaultdict(int)
class_total = defaultdict(int)
for gt, pred in zip(ground_truths, predictions_list):
    class_total[gt] += 1
    if gt in pred or pred in gt:
        class_correct[gt] += 1

print(f"\nPer-class accuracy:")
for cls in sorted(class_total.keys()):
    acc = class_correct[cls] / class_total[cls] * 100
    print(f"  {cls:<20} {class_correct[cls]:>3}/{class_total[cls]:<3} = {acc:.0f}%")

# Show some misclassifications
print(f"\nSample misclassifications:")
misses = [(gt, pred) for gt, pred in zip(ground_truths, predictions_list) if gt not in pred and pred not in gt]
for gt, pred in misses[:10]:
    print(f"  Expected: {gt:<20} Got: {pred}")

In [ ]:
# ---------- Free GPU memory before Model 2 ----------
del model1, trainer1
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"GPU memory freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

---
## 3. Model 2: Qwen-7B LoRA — RCA Generation

| Parameter | Value |
|-----------|-------|
| Base model | `Qwen/Qwen2.5-7B-Instruct` |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Quantization | 4-bit NF4 |
| Target modules | q/k/v/o_proj + gate/up/down_proj |
| Max sequence length | 1024 |

In [ ]:
# ---------- Constants ----------
MODEL2_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL2_OUTDIR = Path("agents/models/trained/model2_qwen")
MAX_SEQ_LEN_2 = 1024

SYSTEM_PROMPT_2 = (
    "You are a Kubernetes Site Reliability Engineering (SRE) agent. "
    "Given raw observability evidence from a Kubernetes incident — including "
    "kubectl output, container logs, and metrics — provide:\n"
    "1. A root cause diagnosis explaining what went wrong and why.\n"
    "2. A step-by-step fix plan to resolve the incident.\n"
    "3. Verification steps to confirm the fix worked."
)

In [ ]:
# ---------- Format data ----------
def build_evidence_prompt(row):
    parts = []
    if row.get("evidence_text", ""):
        parts.append(row["evidence_text"])
    if row.get("scenario_id", ""):
        parts.append(f"\nScenario: {row['scenario_id']}")
    if row.get("difficulty", ""):
        parts.append(f"Difficulty: {row['difficulty']}")
    return "\n".join(parts)

def build_target_response(row):
    sections = []
    if row.get("diagnosis_text", ""):
        sections.append(f"## Diagnosis\n{row['diagnosis_text']}")
    if row.get("fix_plan_text", ""):
        sections.append(f"## Fix Plan\n{row['fix_plan_text']}")
    if row.get("actions_text", ""):
        sections.append(f"## Actions\n{row['actions_text']}")
    if row.get("verification_text", ""):
        sections.append(f"## Verification\n{row['verification_text']}")
    if row.get("rollback_text", ""):
        sections.append(f"## Rollback\n{row['rollback_text']}")
    return "\n\n".join(sections) if sections else "No remediation available."

def build_model2_chat(row):
    evidence = build_evidence_prompt(row)
    target = build_target_response(row)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT_2}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Analyze this Kubernetes incident and provide diagnosis and remediation:\n\n"
        f"{evidence}<|im_end|>\n"
        f"<|im_start|>assistant\n{target}<|im_end|>"
    )

# Prepare dataset
df2 = agent2_df.copy()
str_cols2 = df2.select_dtypes(include=["object"]).columns
df2[str_cols2] = df2[str_cols2].fillna("")
df2 = df2.fillna(0)

texts2 = [build_model2_chat(r) for r in df2.to_dict("records")]
ds2 = Dataset.from_pandas(pd.DataFrame({"text": texts2}))
split2 = ds2.train_test_split(test_size=0.2, seed=42)
print(f"Model 2 — train: {len(split2['train']):,}, test: {len(split2['test']):,}")

lengths = [len(t) for t in texts2]
print(f"Text length — min: {min(lengths)}, max: {max(lengths)}, mean: {sum(lengths)/len(lengths):.0f}")

In [ ]:
# ---------- Load Qwen + LoRA (tuned for higher accuracy) ----------
bnb_config_2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer2 = AutoTokenizer.from_pretrained(MODEL2_ID, trust_remote_code=True)
if tokenizer2.pad_token is None:
    tokenizer2.pad_token = tokenizer2.eos_token
    tokenizer2.pad_token_id = tokenizer2.eos_token_id

model2 = AutoModelForCausalLM.from_pretrained(
    MODEL2_ID,
    quantization_config=bnb_config_2,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model2.config.use_cache = False

# Higher rank + all projection layers for maximum capacity
lora_config_2 = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model2 = get_peft_model(model2, lora_config_2)
trainable, total = model2.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ---------- Train Model 2 (tuned for 70%+ accuracy) ----------
training_args_2 = SFTConfig(
    output_dir=str(MODEL2_OUTDIR),
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    fp16=False,
    report_to="none",
    max_length=MAX_SEQ_LEN_2,
    dataset_text_field="text",
)

trainer2 = SFTTrainer(
    model=model2,
    processing_class=tokenizer2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    args=training_args_2,
)

print("Training Model 2 (Qwen-7B) — 8 epochs, LoRA r=64...")
print("This will take ~30-45 min on A100")
trainer2.train()

In [ ]:
# ---------- Save Model 2 ----------
adapter_path_2 = MODEL2_OUTDIR / "lora_adapter"
model2.save_pretrained(str(adapter_path_2))
tokenizer2.save_pretrained(str(adapter_path_2))

metrics2 = trainer2.evaluate()
with open(MODEL2_OUTDIR / "eval_metrics.json", "w") as f:
    json.dump(metrics2, f, indent=2)

metadata2 = {
    "model_id": MODEL2_ID, "task": "rca_generation",
    "input": "evidence_text", "output": "diagnosis + fix_plan + verification",
    "scenarios": list(agent2_df["scenario_id"].dropna().unique()),
    "lora_r": 16, "lora_alpha": 32, "epochs": 3,
    "dataset_rows": len(ds2),
}
with open(MODEL2_OUTDIR / "model2_metadata.json", "w") as f:
    json.dump(metadata2, f, indent=2)

print(f"\nModel 2 saved to {adapter_path_2}")
print(f"Eval loss: {metrics2.get('eval_loss', 'N/A'):.4f}")

In [ ]:
# ---------- Test Model 2 inference ----------
sample2 = agent2_df.iloc[42].to_dict()
evidence = build_evidence_prompt(sample2)
prompt2 = (
    f"<|im_start|>system\n{SYSTEM_PROMPT_2}<|im_end|>\n"
    f"<|im_start|>user\n"
    f"Analyze this Kubernetes incident and provide diagnosis and remediation:\n\n"
    f"{evidence}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
inputs2 = tokenizer2(prompt2, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN_2)
inputs2 = {k: v.to(model2.device) for k, v in inputs2.items()}

with torch.no_grad():
    outputs2 = model2.generate(**inputs2, max_new_tokens=512, temperature=0.3, do_sample=True, top_p=0.9)
response = tokenizer2.decode(outputs2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)
if "<|im_end|>" in response:
    response = response.split("<|im_end|>")[0]

print(f"Scenario: {sample2['scenario_id']}")
print(f"\n--- Ground Truth ---")
print(sample2.get("diagnosis_text", "")[:300])
print(f"\n--- Model Prediction ---")
print(response.strip())

---
## 4. Download Trained Adapters

Download the LoRA adapters to use locally.

In [ ]:
# Zip and download both adapters
import shutil

shutil.make_archive("model1_mistral_lora", "zip", "agents/models/trained/model1_mistral")
shutil.make_archive("model2_qwen_lora", "zip", "agents/models/trained/model2_qwen")

from google.colab import files
files.download("model1_mistral_lora.zip")
files.download("model2_qwen_lora.zip")
print("Download complete! Extract into agents/models/trained/ locally.")

In [ ]:
# ---------- Summary ----------
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nModel 1 (Mistral-7B) — RCA Classification")
print(f"  Eval loss: {metrics1.get('eval_loss', 'N/A'):.4f}")
print(f"  Adapter:   {adapter_path_1}")
print(f"\nModel 2 (Qwen-7B) — RCA Generation")
print(f"  Eval loss: {metrics2.get('eval_loss', 'N/A'):.4f}")
print(f"  Adapter:   {adapter_path_2}")